# Project 1: Autogen



The overarching goal of this system is to produce a comprehensive and well-structured EDA report.
This report should include a data overview, key insights, visualizations, and a summary of findings. It
must reflect feedback provided by a Critic agent to ensure clarity, accuracy, and actionable insights.
By automating and organizing the EDA process, this framework ensures efficiency, reproducibility,
and high-quality results in data exploration projects.

## Importing Necessary Packages

In [41]:
from autogen import UserProxyAgent, AssistantAgent, GroupChat, GroupChatManager
from autogen.coding import LocalCommandLineCodeExecutor
import os

### Setting up global variables

In [42]:
os.environ['OPENAI_API_KEY'] = "F8kUozKumg8vOqdM6i3uF3MEnHQyAWnh5si5hgocdPdXbakenhTWJQQJ99BLACfhMk5XJ3w3AAAAACOGSVUE"
api_key = os.environ['OPENAI_API_KEY']
api_version = "2024-12-01-preview"
endpoint = "https://genaifoundry766488650611.openai.azure.com/"
model_name = "o3"
deployment = "o3"

### Configuring the LLM and GPT Model

In [43]:
gpt_config = {
    "cache_seed": 42,
    # "temperature": 0.0,
    "config_list": [{"model": model_name, 
                     "api_key": api_key,
                     "api_type": "azure",
                     "base_url": endpoint,
                     "api_version": api_version}],
    "timeout": 120,
}

# Generate the Agents

## Instantiate the Assistant Agents

### Planner Agent

In [44]:
Planner = AssistantAgent(
    name="Planner",
    system_message="""Planner. Suggest a plan. Revise the plan based on feedback
    from admin until admin approval.
    The plan should usually follow the below steps:
    1. Invoke the `Data Preparer` to read and prepare the data. AND execute the code with `Executor`.
    2. Invoke the `EDA` to create the statistics. AND execute the code with `Executor`.
    3. Invoke the `Report Generator` to create the markdown report on the data. AND execute the code with `Executor`.
    4. Invoke the `Critic` to assess the completeness. AND execute the code with `Executor`.
    5. Go back to the Admin to get their approval
    6. Repeat steps 1 to 5 until approval
    """,
    llm_config=gpt_config
)

### Data Preparer Agent

In [45]:
DataPreparer = AssistantAgent(
    name="DataPreparer",
    llm_config=gpt_config,
    system_message="A Data Reader, Cleaner and Preprocessor. \
    your first role is to load the dataset (csv, excel, parquet)\
    after loading the data you handle missing values and duplicates. \
    Provide a data overview consisting of a table of the column names and datatypes along with a print of the first 10 rows.",
)

### EDA Agent

In [46]:
EDA = AssistantAgent(
    name="EDA",
    llm_config=gpt_config,
    system_message="A Data Explorer and Analyzer. \
    Generate statistical summaries (mean, median, distributions) \
    Create visualizations (histograms, scatter plots, correlation heatmaps). \
    Identify patterns, anomalies, and relationships in the data.\
    Extract key insights for decision‑making.",
)

### Code Writer

In [47]:
CodeWriter = AssistantAgent(
    name="CodeWriter",
    llm_config=gpt_config,
    system_message="write the codes generated by DataPreparer and EDA as an ipynb script",
)

### Report Generator Agent

In [48]:
ReportGenerator = AssistantAgent(
    name="ReportGenerator",
    llm_config=gpt_config,
    system_message="A Documentation and reporting specialist \
    Always output Python code that writes the final report to a file called\
    './data/churn_prediction_report.md'\
    Organize sections: a data overview, key insights, visualizations, and a summary of findings.\
    Ensure readability and professional formatting.",
)


### Critic Agent

In [49]:
Critic = AssistantAgent(
    name="Critic",
    llm_config=gpt_config,
    system_message="Reviewer and quality controller. \
    Review the draft report for clarity, accuracy, and completeness.\
    Flag unclear explanations, misleading visuals, or missing insights.\
    Provide constructive feedback to improve the report."
)

## Intantiate the UserProxy Agents

### Admin Agent - UserProxy

In [50]:
user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss\
        the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False
)

### Executor Agent

In [51]:
Executor = UserProxyAgent(
    name="Executor",
    system_message="Code runner and validator.\
        Execute Python code generated by the AssistantAgents\
        Validate outputs (e.g., checks if plots render, summaries compute correctly).\
        Return execution results to the assistants for refinement.\
        Ensure reproducibility and correctness of the workflow.",
    human_input_mode="ALWAYS",
    code_execution_config={"last_n_messages": 3,
                           "work_dir": ".",
                           "use_docker": False},
)

# Defining the Workflow

In [52]:
groupchat = GroupChat(
    agents=[user_proxy, DataPreparer, EDA, CodeWriter, ReportGenerator, Critic, Executor, Planner], 
    messages=[], 
    max_round=5
)

In [53]:
manager = GroupChatManager(groupchat=groupchat, 
                           llm_config=gpt_config)

In [56]:
user_proxy.initiate_chat(
    manager,
    message="""
    initiate data prepro then EDA on './data/nyc_taxi_trip_duration.csv'
    You compile results into a structured report.
    write the python code as ./data/pipeline.ipynb .
    Always output Python code that saves the final report as './data/nyc_taxi_trip_duration.md'.""",)

Admin (to chat_manager):


    initiate data prepro then EDA on './data/nyc_taxi_trip_duration.csv'
    You compile results into a structured report.
    write the python code as ./data/pipeline.ipynb .
    Always output Python code that saves the final report as './data/nyc_taxi_trip_duration.md'.

--------------------------------------------------------------------------------
[autogen.oai.client: 02-17 11:44:39] {329} WARNING - Model o3-2025-04-16 is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.

Next speaker: Planner

[autogen.oai.client: 02-17 11:44:48] {329} WARNING - Model o3-2025-04-16 is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
Planner (to chat_manager):

Proposed Plan for “nyc_taxi_trip_duration” Analysis Pipeline
Data source: “./data/nyc_taxi_trip_duration.csv”  


ChatResult(chat_id=None, chat_history=[{'content': "\n    initiate data prepro then EDA on './data/nyc_taxi_trip_duration.csv'\n    You compile results into a structured report.\n    write the python code as ./data/pipeline.ipynb .\n    Always output Python code that saves the final report as './data/nyc_taxi_trip_duration.md'.", 'role': 'assistant', 'name': 'Admin'}, {'content': 'Proposed Plan for “nyc_taxi_trip_duration” Analysis Pipeline\n================================================================\nData source: “./data/nyc_taxi_trip_duration.csv”  \nDesired deliverables:  \n• Jupyter notebook saved as “./data/pipeline.ipynb” (contains all steps & code)  \n• Markdown report saved as “./data/nyc_taxi_trip_duration.md”\n\nStep-by-Step Workflow\n---------------------\n1. Data Preparer  \n   a. Load the CSV.  \n   b. Basic data‐cleaning: handle missing values, convert date/time fields, create trip_duration (if absent), drop obvious outliers, set correct dtypes, save a “clean” DataFr